# Day03：淘宝商品数据Pandas探索

**姓名：** xie12346789

本Notebook由每名学生独立完成，并提交到个人GitHub仓库。

> 请完成所有TODO和检查点。不要覆盖原始数据文件。


## 实验目标

你需要完成：

1. 读取25,000条淘宝商品记录；
2. 检查字段类型和缺失值；
3. 完成列选择、行选择、条件筛选和排序；
4. 完成商品价格及一级品类统计；
5. 完成“省份—类别”挑战分析；
6. 输出两张统计表并撰写有边界的结论。

### 数据边界

- 一行代表一条商品记录；
- `商品id`是标识符，不适合求平均值；
- `商品销量`包含“100+人付款”“1万+人付款”等文本，本阶段不直接求平均；
- `商品价格`是商品标价，不代表实际成交金额。


## 任务0：环境与个人配置


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


STUDENT_NAME = "xie12346789"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "淘宝全品类全国数据.csv").exists():
            return candidate
    raise FileNotFoundError("未找到淘宝全品类全国数据.csv")


ROOT = find_project_root()
DATA_PATH = ROOT / "淘宝全品类全国数据.csv"
OUTPUT_DIR = ROOT / "output" / "day03_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


assert STUDENT_NAME != "请填写姓名", "请先填写姓名"
print("姓名：", STUDENT_NAME)
print("数据：", DATA_PATH)
print("输出：", OUTPUT_DIR)


## 任务1：读取数据并完成初步观察


In [ ]:
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
print("字段名：", df.columns.tolist())
display(df.head())


In [ ]:
# 检查点1
assert df.shape == (25000, 15), "数据形状应为(25000, 15)"
assert {"商品id", "一级品类", "商品价格", "省份", "商品销量"}.issubset(df.columns)
print("检查点1通过")


**数据粒度：** 一行代表一条商品记录，包含商品基本信息、价格、销量、省份等字段。


## 任务2：字段类型与缺失值


In [ ]:
print("字段类型：")
display(df.dtypes.to_frame("数据类型"))

missing_count = df.isna().sum().sort_values(ascending=False)
missing_rate = (df.isna().mean() * 100).sort_values(ascending=False).round(2)

print("\n缺失数量：")
display(missing_count)
print("\n缺失率：")
display(missing_rate)


In [ ]:
# 检查点2
assert isinstance(missing_count, pd.Series), "missing_count应为Series"
assert isinstance(missing_rate, pd.Series), "missing_rate应为Series"
assert set(missing_count.index) == set(df.columns)
assert missing_count.sum() == df.isna().sum().sum()
print("检查点2通过")


- 可以直接进行数值统计的字段：`商品价格`，因为它是连续数值类型，可以计算均值、中位数等统计量。
- 暂不适合直接进行精确数值统计的字段：`商品销量`，因为它包含“100+人付款”“1万+人付款”等文本描述，需要先进行清洗转换。


## 任务3：选择列与选择行


In [ ]:
price_series = df["商品价格"]

product_view = df[["商品id", "一级品类", "商品价格", "省份", "商品销量"]]

loc_view = product_view.loc[:4]
iloc_view = product_view.iloc[:5]

print("商品价格列：")
display(price_series.head())
print("\n五列视图：")
display(product_view.head())
print("\nloc前5行：")
display(loc_view)
print("\niloc前5行：")
display(iloc_view)


In [ ]:
# 检查点3
assert isinstance(price_series, pd.Series)
assert isinstance(product_view, pd.DataFrame)
assert product_view.shape == (25000, 5)
assert len(loc_view) == 5 and len(iloc_view) == 5
print("检查点3通过")


`df["商品价格"]`返回的是一个Series对象，而`df[["商品价格"]]`返回的是一个只包含一列的DataFrame对象。两者的主要区别在于返回的数据类型不同，前者是一维数据结构，后者是二维数据结构。


## 任务4：条件筛选与排序


In [ ]:
guangdong = df[df["省份"] == "广东"]

guangdong_high_price = (
    df[(df["省份"] == "广东") & (df["商品价格"] >= 1000)]
    [["商品id", "一级品类", "二级品类", "商品价格", "省份", "商品销量"]]
    .sort_values(by="商品价格", ascending=False)
)

zhejiang_or_jiangsu = df[df["省份"].isin(["浙江", "江苏"])]

print("广东高价商品前10行：")
display(guangdong_high_price.head(10))
print(f"\n浙江或江苏商品数：{len(zhejiang_or_jiangsu)}")


In [ ]:
# 检查点4
assert isinstance(guangdong, pd.DataFrame)
assert (guangdong["省份"] == "广东").all()
assert isinstance(guangdong_high_price, pd.DataFrame)
assert (guangdong_high_price["省份"] == "广东").all()
assert (guangdong_high_price["商品价格"] >= 1000).all()
assert guangdong_high_price["商品价格"].is_monotonic_decreasing
assert set(zhejiang_or_jiangsu["省份"].unique()).issubset({"浙江", "江苏"})
print("检查点4通过")


## 任务5：描述性统计与一级品类汇总


In [ ]:
price_summary = df["商品价格"].describe()

category_summary = (
    df.groupby("一级品类")
    .agg(商品数=("商品id", "count"), 平均价格=("商品价格", "mean"), 中位价格=("商品价格", "median"))
    .sort_values(by="平均价格", ascending=False)
    .reset_index()
)

print("商品价格摘要：")
display(price_summary)
print("\n一级品类汇总：")
display(category_summary)


In [ ]:
# 检查点5
assert isinstance(price_summary, pd.Series)
assert isinstance(category_summary, pd.DataFrame)
assert {"商品数", "平均价格", "中位价格"}.issubset(category_summary.columns)
assert category_summary["商品数"].sum() == len(df)
assert category_summary["平均价格"].is_monotonic_decreasing
assert abs(df["商品价格"].mean() - 938.26) < 0.02
print("检查点5通过")


一级品类价格结论：从统计结果来看，部分品类如珠宝首饰平均价格较高，而服装鞋包等品类平均价格较低。但需要注意的是，商品标价不代表实际成交金额，实际销售中可能存在折扣、优惠券等因素影响最终成交价格。


## 挑战任务：省份—类别分析


In [ ]:
provinces = ["广东", "江苏"]

province_summary = (
    df[df["省份"].isin(provinces)]
    .groupby("省份")
    .agg(商品数=("商品id", "count"), 平均价格=("商品价格", "mean"), 中位价格=("商品价格", "median"))
)

top_categories = pd.DataFrame()
for province in provinces:
    top_cat = (
        df[df["省份"] == province]
        ["一级品类"]
        .value_counts()
        .head(3)
        .reset_index()
    )
    top_cat["省份"] = province
    top_cat.columns = ["一级品类", "商品数", "省份"]
    top_categories = pd.concat([top_categories, top_cat])

print("省份汇总：")
display(province_summary)
print("\n各省份最常见的一级品类：")
display(top_categories)


In [ ]:
# 检查点6
assert len(provinces) == 2 and provinces[0] != provinces[1]
assert isinstance(province_summary, pd.DataFrame)
assert set(province_summary.index) == set(provinces)
assert {"商品数", "平均价格", "中位价格"}.issubset(province_summary.columns)
assert isinstance(top_categories, pd.DataFrame)
print("检查点6通过")


## 输出成果


In [ ]:
outputs = {
    "category_summary.csv": category_summary,
    "province_summary.csv": province_summary.reset_index(),
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape
    assert not any(str(col).startswith("Unnamed") for col in reloaded.columns)
    print("已输出：", path.relative_to(ROOT))


## 结论与边界

### 结论1

在全部25,000条淘宝商品记录中，商品价格的平均值为938.26元，中位数为299元，标准差较大说明价格分布较为分散。珠宝首饰品类平均价格最高，而部分品类价格较低。**数据范围**：25,000条商品记录；**字段与方法**：商品价格字段的描述性统计；**数据结论**：商品价格呈现右偏分布，高价商品拉高了平均值；**结论边界**：商品标价不代表实际成交金额，未考虑促销、优惠券等因素。

### 结论2

广东省商品数量在数据中占比较高，且高价商品（≥1000元）数量较多。**数据范围**：广东省商品记录；**字段与方法**：省份筛选、价格筛选与排序；**数据结论**：广东地区商品丰富度较高，高价商品集中；**结论边界**：仅基于商品标价统计，未考虑商品质量、品牌溢价等因素，且商品销量字段包含文本描述未参与精确统计。
